# House Price Prediction using Linear Regression
## Predictive Modeling Project

**Objective:** Build, evaluate, and interpret a Linear Regression model to predict house prices.

**Dataset:** Real-world house price data with multiple features (square footage, bedrooms, bathrooms, location, etc.)

**Deliverables:**
- Model with MAE, MSE, and R² metrics
- Feature importance analysis
- Price predictions on test data

## Step 0: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## Step 1: Dataset Selection & Loading
Load the house price prediction dataset and display initial information.

In [ ]:
# Load the training dataset
train_data = pd.read_csv('train.csv.zip')
test_data = pd.read_csv('test.csv.zip')

print("Dataset loaded successfully!")
print(f"\nTraining data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")
print(f"\nFirst few rows of training data:")
train_data.head()

In [ ]:
# Display data types
print("Data types in training data:")
print(train_data.dtypes)
print(f"\nColumn names: {train_data.columns.tolist()}")

## Step 2: Exploratory Data Analysis (EDA)
Understand the data through statistics and visualizations.

In [ ]:
# Check for missing values
print("Missing values in training data:")
missing_values = train_data.isnull().sum()
print(missing_values[missing_values > 0])
print(f"\nTotal missing values: {train_data.isnull().sum().sum()}")

In [ ]:
# Summary statistics for numerical features
print("Summary Statistics:")
train_data.describe()

In [ ]:
# Identify the target variable (typically named 'SalePrice' or 'Price')
# Adjust the column name based on your actual dataset
target_column = 'SalePrice'  # Change this if your dataset uses a different name

# Check if target column exists
if target_column in train_data.columns:
    print(f"Target variable found: {target_column}")
else:
    print(f"Available columns: {train_data.columns.tolist()}")
    print("Please identify the target column (price/sale price column)")

In [ ]:
# Visualization 1: Distribution of Target Variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(train_data[target_column], bins=50, color='skyblue', edgecolor='black')
axes[0].set_title('Distribution of House Prices', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(train_data[target_column])
axes[1].set_title('Box Plot of House Prices', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Price ($)', fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Price: ${train_data[target_column].mean():,.2f}")
print(f"Median Price: ${train_data[target_column].median():,.2f}")
print(f"Std Dev: ${train_data[target_column].std():,.2f}")

In [ ]:
# Visualization 2: Correlation Heatmap
# Select only numerical columns for correlation analysis
numerical_columns = train_data.select_dtypes(include=[np.number]).columns.tolist()

# Create correlation matrix
correlation_matrix = train_data[numerical_columns].corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show correlations with target variable
print(f"\nCorrelations with {target_column}:")
target_correlations = correlation_matrix[target_column].sort_values(ascending=False)
print(target_correlations)

In [ ]:
# Visualization 3: Scatter Plots of Top Features vs Target
# Get top 4 features (excluding target itself)
top_features = target_correlations[1:5].index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, feature in enumerate(top_features):
    axes[idx].scatter(train_data[feature], train_data[target_column], alpha=0.5, color='steelblue')
    axes[idx].set_xlabel(feature, fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(target_column, fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{feature} vs {target_column}', fontsize=12, fontweight='bold')
    axes[idx].grid(alpha=0.3)
    
    # Add correlation coefficient
    correlation = train_data[feature].corr(train_data[target_column])
    axes[idx].text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
                   transform=axes[idx].transAxes, fontsize=10,
                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Step 3: Data Preprocessing
Prepare the data for Linear Regression model.

In [ ]:
# Create a copy of training data for preprocessing
data = train_data.copy()

print("Step 3.1: Handle Missing Values")
print(f"Missing values before imputation:\n{data.isnull().sum()[data.isnull().sum() > 0]}")

# Strategy: Drop columns with too many missing values (>50%), impute others
missing_percentage = (data.isnull().sum() / len(data)) * 100
columns_to_drop = missing_percentage[missing_percentage > 50].index.tolist()

if columns_to_drop:
    print(f"\nColumns to drop (>50% missing): {columns_to_drop}")
    data = data.drop(columns=columns_to_drop)

# Impute numerical missing values with median
numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if target_column in numerical_cols:
    numerical_cols.remove(target_column)

for col in numerical_cols:
    if data[col].isnull().sum() > 0:
        median_value = data[col].median()
        data[col].fillna(median_value, inplace=True)
        print(f"Imputed {col} with median: {median_value}")

# Impute categorical missing values with mode
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    if data[col].isnull().sum() > 0:
        mode_value = data[col].mode()[0]
        data[col].fillna(mode_value, inplace=True)
        print(f"Imputed {col} with mode: {mode_value}")

print(f"\nMissing values after imputation: {data.isnull().sum().sum()}")

In [ ]:
# Step 3.2: Encode Categorical Variables
print("Step 3.2: Encode Categorical Variables\n")

# Identify categorical columns
categorical_columns = data.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_columns}")

# One-Hot Encoding for categorical variables
if categorical_columns:
    data_encoded = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"\nShape after encoding: {data_encoded.shape}")
    print(f"New columns created: {len(data_encoded.columns) - len(data.columns)}")
else:
    data_encoded = data.copy()
    print("No categorical columns to encode.")

In [ ]:
# Step 3.3: Feature Selection (Remove low correlation features)
print("Step 3.3: Feature Selection\n")

# Separate features and target
X = data_encoded.drop(columns=[target_column])
y = data_encoded[target_column]

# Calculate correlation with target
correlations_with_target = X.corrwith(y).abs().sort_values(ascending=False)

print(f"Features with correlation < 0.05 (potential removal):")
low_corr_features = correlations_with_target[correlations_with_target < 0.05].index.tolist()
print(f"Count: {len(low_corr_features)}")

# Remove extremely low correlation features (optional - keep for this project)
# X = X.drop(columns=low_corr_features)

print(f"\nFinal feature set shape: {X.shape}")
print(f"Top 10 features by correlation with {target_column}:")
print(correlations_with_target.head(10))

In [ ]:
# Step 3.4: Train-Test Split (80% Train, 20% Test)
print("Step 3.4: Train-Test Split\n")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Number of features: {X_train.shape[1]}")
print(f"\nTraining target statistics:")
print(f"  Mean: ${y_train.mean():,.2f}")
print(f"  Std Dev: ${y_train.std():,.2f}")
print(f"\nTest target statistics:")
print(f"  Mean: ${y_test.mean():,.2f}")
print(f"  Std Dev: ${y_test.std():,.2f}")

## Step 4: Model Training
Train the Linear Regression model on the training data.

In [ ]:
# Initialize and train the Linear Regression model
print("Step 4: Model Training\n")

model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression model trained successfully!")
print(f"Model Intercept: ${model.intercept_:,.2f}")
print(f"Number of features: {len(model.coef_)}")

In [ ]:
# Make predictions on both training and test sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("Predictions made on both training and test sets.")
print(f"\nFirst 10 predictions (Test Set):")
comparison = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_test_pred[:10],
    'Difference': y_test.values[:10] - y_test_pred[:10]
})
comparison

## Step 5: Model Evaluation
Calculate and interpret key performance metrics.

In [ ]:
# Calculate evaluation metrics
print("Step 5: Model Evaluation\n")
print("="*60)
print("TRAINING SET METRICS")
print("="*60)

# Training metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

print(f"Mean Absolute Error (MAE):  ${train_mae:,.2f}")
print(f"Mean Squared Error (MSE):   ${train_mse:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${train_rmse:,.2f}")
print(f"R-squared Score (R²):       {train_r2:.4f}")
print(f"\nInterpretation: The model explains {train_r2*100:.2f}% of the variance in house prices.")

In [ ]:
# Test metrics
print("\n" + "="*60)
print("TEST SET METRICS")
print("="*60)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Mean Absolute Error (MAE):  ${test_mae:,.2f}")
print(f"Mean Squared Error (MSE):   ${test_mse:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${test_rmse:,.2f}")
print(f"R-squared Score (R²):       {test_r2:.4f}")
print(f"\nInterpretation: The model explains {test_r2*100:.2f}% of the variance in test house prices.")

In [ ]:
# Model Comparison: Training vs Test
print("\n" + "="*60)
print("TRAINING VS TEST COMPARISON")
print("="*60)

metrics_comparison = pd.DataFrame({
    'Training': [train_mae, train_mse, train_rmse, train_r2],
    'Test': [test_mae, test_mse, test_rmse, test_r2],
    'Metric': ['MAE ($)', 'MSE ($)', 'RMSE ($)', 'R² Score']
})
metrics_comparison = metrics_comparison.set_index('Metric')
print(metrics_comparison)

# Check for overfitting
r2_diff = train_r2 - test_r2
if r2_diff > 0.1:
    print(f"\n⚠️  Warning: Possible overfitting (R² diff: {r2_diff:.4f})")
else:
    print(f"\n✓ Model is well-generalized (R² diff: {r2_diff:.4f})")

In [ ]:
# Visualization: Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Set
axes[0].scatter(y_train, y_train_pred, alpha=0.5, color='steelblue')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price ($)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predicted Price ($)', fontsize=12, fontweight='bold')
axes[0].set_title(f'Training Set: Actual vs Predicted (R² = {train_r2:.4f})', 
                  fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Test Set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, color='coral')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Price ($)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Predicted Price ($)', fontsize=12, fontweight='bold')
axes[1].set_title(f'Test Set: Actual vs Predicted (R² = {test_r2:.4f})', 
                  fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residuals Analysis
train_residuals = y_train - y_train_pred
test_residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Residuals
axes[0].scatter(y_train_pred, train_residuals, alpha=0.5, color='steelblue')
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Price ($)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Residuals ($)', fontsize=12, fontweight='bold')
axes[0].set_title('Training Set: Residual Plot', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)

# Test Residuals
axes[1].scatter(y_test_pred, test_residuals, alpha=0.5, color='coral')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price ($)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Residuals ($)', fontsize=12, fontweight='bold')
axes[1].set_title('Test Set: Residual Plot', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean of training residuals: ${train_residuals.mean():,.2f}")
print(f"Mean of test residuals: ${test_residuals.mean():,.2f}")

## Step 6: Interpretation & Feature Importance
Extract coefficients and explain feature impact on predictions.

In [ ]:
# Extract model coefficients
print("Step 6: Model Interpretation\n")
print("="*70)
print("MODEL COEFFICIENTS & FEATURE IMPORTANCE")
print("="*70)

# Create a DataFrame of coefficients
coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

# Sort by absolute value of coefficient
coefficients_df['Abs_Coefficient'] = coefficients_df['Coefficient'].abs()
coefficients_df = coefficients_df.sort_values('Abs_Coefficient', ascending=False)

print(f"\nModel Intercept: ${model.intercept_:,.2f}")
print("\nTop 15 Most Important Features:")
print(coefficients_df.head(15)[['Feature', 'Coefficient']])

In [ ]:
# Visualization: Top Features Coefficient Plot
top_n = 15
top_features_coef = coefficients_df.head(top_n)

plt.figure(figsize=(10, 8))
colors = ['green' if x > 0 else 'red' for x in top_features_coef['Coefficient']]
plt.barh(range(len(top_features_coef)), top_features_coef['Coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(top_features_coef)), top_features_coef['Feature'])
plt.xlabel('Coefficient Value', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title(f'Top {top_n} Features by Coefficient Magnitude', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Green bars = Positive correlation (increase feature → increase price)")
print("Red bars = Negative correlation (increase feature → decrease price)")

In [ ]:
# Interpretation of top features
print("\nFEATURE IMPACT INTERPRETATION:")
print("="*70)

for idx, row in coefficients_df.head(10).iterrows():
    feature = row['Feature']
    coef = row['Coefficient']
    
    if coef > 0:
        direction = "increases"
    else:
        direction = "decreases"
    
    print(f"\n{feature}:")
    print(f"  Coefficient: {coef:,.2f}")
    print(f"  Impact: For a 1-unit increase in {feature}, the price {direction} by ${abs(coef):,.2f}")

In [ ]:
# Generate predictions for the actual test data
print("\nGeneating predictions for test dataset...\n")

# Preprocess test data the same way as training data
test_data_processed = test_data.copy()

# Apply the same transformations (imputation, encoding, etc.)
# Drop the same columns
if columns_to_drop:
    test_data_processed = test_data_processed.drop(columns=[col for col in columns_to_drop if col in test_data_processed.columns])

# Impute missing values
test_numerical_cols = test_data_processed.select_dtypes(include=[np.number]).columns.tolist()
for col in test_numerical_cols:
    if test_data_processed[col].isnull().sum() > 0:
        median_value = test_data_processed[col].median()
        test_data_processed[col].fillna(median_value, inplace=True)

test_categorical_cols = test_data_processed.select_dtypes(include=['object']).columns.tolist()
for col in test_categorical_cols:
    if test_data_processed[col].isnull().sum() > 0:
        mode_value = test_data_processed[col].mode()[0]
        test_data_processed[col].fillna(mode_value, inplace=True)

# One-Hot Encoding
test_data_processed = pd.get_dummies(test_data_processed, columns=test_categorical_cols, drop_first=True)

# Align columns with training data
missing_cols = set(X.columns) - set(test_data_processed.columns)
for col in missing_cols:
    test_data_processed[col] = 0

# Ensure same column order
X_test_full = test_data_processed[X.columns]

# Make predictions
test_predictions = model.predict(X_test_full)

print(f"Predictions generated for {len(test_predictions)} test samples")
print(f"\nFirst 10 predictions:")
print(test_predictions[:10])

In [ ]:
# Create submission file
if 'Id' in test_data.columns:
    submission = pd.DataFrame({
        'Id': test_data['Id'],
        'SalePrice': test_predictions
    })
elif 'ID' in test_data.columns:
    submission = pd.DataFrame({
        'ID': test_data['ID'],
        'SalePrice': test_predictions
    })
else:
    submission = pd.DataFrame({
        'SalePrice': test_predictions
    })
    print("Note: Could not find ID column. Check test data format.")

print("\nSubmission Preview:")
print(submission.head(10))

# Save submission
submission.to_csv('predictions.csv', index=False)
print(f"\n✓ Predictions saved to 'predictions.csv'")

## Summary & Conclusion
Model performance overview and real-world applicability assessment.

In [ ]:
# Final Summary
print("\n" + "="*70)
print("PROJECT SUMMARY & CONCLUSIONS")
print("="*70)

print(f"\n✓ DATASET:")
print(f"  • Total samples: {len(train_data)}")
print(f"  • Features used: {X.shape[1]}")
print(f"  • Target variable: {target_column}")
print(f"  • Price range: ${y.min():,.0f} - ${y.max():,.0f}")

print(f"\n✓ MODEL PERFORMANCE:")
print(f"  • Training R²: {train_r2:.4f} ({train_r2*100:.2f}%)")
print(f"  • Test R²: {test_r2:.4f} ({test_r2*100:.2f}%)")
print(f"  • Test MAE: ${test_mae:,.2f}")
print(f"  • Test RMSE: ${test_rmse:,.2f}")
print(f"  • Average error: {(test_mae/y_test.mean()*100):.2f}% of mean price")

print(f"\n✓ TOP 3 PREDICTIVE FEATURES:")
for i, (idx, row) in enumerate(coefficients_df.head(3).iterrows(), 1):
    print(f"  {i}. {row['Feature']} (coef: {row['Coefficient']:,.2f})")

print(f"\n✓ MODEL ASSESSMENT:")
if test_r2 > 0.8:
    accuracy = "Excellent (>0.8)"
elif test_r2 > 0.7:
    accuracy = "Good (0.7-0.8)"
elif test_r2 > 0.6:
    accuracy = "Moderate (0.6-0.7)"
else:
    accuracy = "Poor (<0.6)"
    
print(f"  • Model Accuracy: {accuracy}")
print(f"  • Generalization: {'Good ✓' if r2_diff < 0.1 else 'Possible Overfitting ⚠'}")
print(f"  • Real-world Application: {'Recommended ✓' if test_r2 > 0.7 else 'Needs Improvement'}")

print(f"\n✓ LIMITATIONS:")
print(f"  • Model assumes linear relationships between features and price")
print(f"  • May not capture complex interactions between features")
print(f"  • Performance depends heavily on data quality and feature engineering")
print(f"  • Model should be retrained with new data periodically")

print(f"\n" + "="*70)

In [ ]:
# Final Statistics Table
print("\n📊 FINAL STATISTICS TABLE\n")

final_summary = pd.DataFrame({
    'Metric': [
        'Training R² Score',
        'Test R² Score',
        'Training MAE',
        'Test MAE',
        'Training RMSE',
        'Test RMSE',
        'Total Features',
        'Training Samples',
        'Test Samples',
        'Model Intercept'
    ],
    'Value': [
        f'{train_r2:.4f}',
        f'{test_r2:.4f}',
        f'${train_mae:,.2f}',
        f'${test_mae:,.2f}',
        f'${train_rmse:,.2f}',
        f'${test_rmse:,.2f}',
        f'{X.shape[1]}',
        f'{len(X_train)}',
        f'{len(X_test)}',
        f'${model.intercept_:,.2f}'
    ]
})

print(final_summary.to_string(index=False))